# HR Workforce Analytics — Data Cleaning & Transformation

This notebook performs data cleaning and transformation on the raw
HR Employee Attrition dataset.

## Objectives

- Remove redundant columns
- Validate column names and data types
- Validate categorical and ordinal values
- Validate business logic
- Prepare analytical features
- Perform final data quality checks
- Export the cleaned dataset for PostgreSQL, SQL analysis,
  Python EDA, and Power BI

The raw dataset will not be modified.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
RAW_PATH = Path("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")
PROCESSED_DIR = Path("../data/processed")
OUTPUT_PATH = PROCESSED_DIR / "hr_employee_attrition_clean.csv"

In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
df = pd.read_csv(RAW_PATH)

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [8]:
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Rows    : 1,470
Columns : 35


## Cleaning Strategy

Based on the data quality assessment performed in Phase 2, the dataset
contains no missing values, duplicate rows, invalid ranges, or logical
inconsistencies.

Therefore, the cleaning process will focus primarily on structural
simplification, standardization, and analytical preparation.

### Columns to Remove

The following constant/redundant columns will be removed:

- `EmployeeCount`
- `Over18`
- `StandardHours`

These columns contain only one unique value and do not provide
meaningful analytical variation.

### Columns to Keep

`EmployeeNumber` will be retained as the unique employee identifier.

All other variables will be retained because they may provide useful
information for workforce, performance, compensation, satisfaction,
tenure, and attrition analysis.

### Outlier Treatment

Statistical outliers identified in Phase 2 will not be removed
automatically because they represent valid business observations.

### Missing Values

No missing-value treatment is required because the dataset contains
no missing values.

### Duplicate Rows

No duplicate removal is required because the dataset contains no
duplicate rows.

In [9]:
columns_to_remove = [
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_clean = df.drop(columns=columns_to_remove).copy()

In [10]:
print(f"Original columns : {df.shape[1]}")
print(f"Cleaned columns  : {df_clean.shape[1]}")

Original columns : 35
Cleaned columns  : 32


In [11]:
set(columns_to_remove).intersection(df_clean.columns)

set()

In [12]:
print(df_clean.columns.tolist())

['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [13]:
employee_id_unique = df_clean["EmployeeNumber"].is_unique

print(f"EmployeeNumber unique: {employee_id_unique}")

EmployeeNumber unique: True


In [14]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.replace(r"(?<!^)(?=[A-Z])", "_", regex=True)
    .str.lower()
)

In [15]:
df_clean.columns.tolist()

['age',
 'attrition',
 'business_travel',
 'daily_rate',
 'department',
 'distance_from_home',
 'education',
 'education_field',
 'employee_number',
 'environment_satisfaction',
 'gender',
 'hourly_rate',
 'job_involvement',
 'job_level',
 'job_role',
 'job_satisfaction',
 'marital_status',
 'monthly_income',
 'monthly_rate',
 'num_companies_worked',
 'over_time',
 'percent_salary_hike',
 'performance_rating',
 'relationship_satisfaction',
 'stock_option_level',
 'total_working_years',
 'training_times_last_year',
 'work_life_balance',
 'years_at_company',
 'years_in_current_role',
 'years_since_last_promotion',
 'years_with_curr_manager']

## Data Type Standardization

The cleaned dataset contains numerical, categorical, and ordinal variables.

Data types will be reviewed and standardized before the dataset is
exported and loaded into PostgreSQL.

In [16]:
df_clean.dtypes

age                           int64
attrition                       str
business_travel                 str
daily_rate                    int64
department                      str
distance_from_home            int64
education                     int64
education_field                 str
employee_number               int64
environment_satisfaction      int64
gender                          str
hourly_rate                   int64
job_involvement               int64
job_level                     int64
job_role                        str
job_satisfaction              int64
marital_status                  str
monthly_income                int64
monthly_rate                  int64
num_companies_worked          int64
over_time                       str
percent_salary_hike           int64
performance_rating            int64
relationship_satisfaction     int64
stock_option_level            int64
total_working_years           int64
training_times_last_year      int64
work_life_balance           

In [17]:
categorical_columns = [
    "attrition",
    "business_travel",
    "department",
    "education_field",
    "gender",
    "job_role",
    "marital_status",
    "over_time"
]

In [18]:
df_clean[categorical_columns].dtypes

attrition          str
business_travel    str
department         str
education_field    str
gender             str
job_role           str
marital_status     str
over_time          str
dtype: object

In [19]:
for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df_clean[column].value_counts(dropna=False))


--- attrition ---
attrition
No     1233
Yes     237
Name: count, dtype: int64

--- business_travel ---
business_travel
Travel_Rarely        1043
Travel_Frequently     277
Non-Travel            150
Name: count, dtype: int64

--- department ---
department
Research & Development    961
Sales                     446
Human Resources            63
Name: count, dtype: int64

--- education_field ---
education_field
Life Sciences       606
Medical             464
Marketing           159
Technical Degree    132
Other                82
Human Resources      27
Name: count, dtype: int64

--- gender ---
gender
Male      882
Female    588
Name: count, dtype: int64

--- job_role ---
job_role
Sales Executive              326
Research Scientist           292
Laboratory Technician        259
Manufacturing Director       145
Healthcare Representative    131
Manager                      102
Sales Representative          83
Research Director             80
Human Resources               52
Name: count, dtyp

In [20]:
expected_categories = {
    "attrition": {"Yes", "No"},
    "business_travel": {
        "Travel_Rarely",
        "Travel_Frequently",
        "Non-Travel"
    },
    "department": {
        "Research & Development",
        "Sales",
        "Human Resources"
    },
    "education_field": {
        "Life Sciences",
        "Medical",
        "Marketing",
        "Technical Degree",
        "Other",
        "Human Resources"
    },
    "gender": {
        "Male",
        "Female"
    },
    "job_role": {
        "Sales Executive",
        "Research Scientist",
        "Laboratory Technician",
        "Manufacturing Director",
        "Healthcare Representative",
        "Manager",
        "Sales Representative",
        "Research Director",
        "Human Resources"
    },
    "marital_status": {
        "Married",
        "Single",
        "Divorced"
    },
    "over_time": {
        "Yes",
        "No"
    }
}

In [21]:
category_validation = []

for column, expected in expected_categories.items():
    actual = set(df_clean[column].dropna().unique())

    unexpected = actual - expected
    missing_expected = expected - actual

    category_validation.append({
        "column": column,
        "actual_categories": sorted(actual),
        "unexpected_categories": sorted(unexpected),
        "missing_expected_categories": sorted(missing_expected),
        "valid": len(unexpected) == 0
    })

category_validation = pd.DataFrame(category_validation)

category_validation

,column,actual_categories,unexpected_categories,missing_expected_categories,valid
0,attrition,"[No, Yes]",[],[],True
1,business_travel,"[Non-Travel, Travel_Frequently, Travel_Rarely]",[],[],True
2,department,"[Human Resources, Research & Development, Sales]",[],[],True
3,education_field,"[Human Resources, Life Sciences, Marketing, Me...",[],[],True
4,gender,"[Female, Male]",[],[],True
5,job_role,"[Healthcare Representative, Human Resources, L...",[],[],True
6,marital_status,"[Divorced, Married, Single]",[],[],True
7,over_time,"[No, Yes]",[],[],True


In [22]:
ordinal_columns = [
    "education",
    "environment_satisfaction",
    "job_involvement",
    "job_level",
    "job_satisfaction",
    "performance_rating",
    "relationship_satisfaction",
    "stock_option_level",
    "work_life_balance"
]

In [23]:
df_clean[ordinal_columns].dtypes

education                    int64
environment_satisfaction     int64
job_involvement              int64
job_level                    int64
job_satisfaction             int64
performance_rating           int64
relationship_satisfaction    int64
stock_option_level           int64
work_life_balance            int64
dtype: object

In [24]:
ordinal_ranges = {
    "education": (1, 5),
    "environment_satisfaction": (1, 4),
    "job_involvement": (1, 4),
    "job_level": (1, 5),
    "job_satisfaction": (1, 4),
    "performance_rating": (3, 4),
    "relationship_satisfaction": (1, 4),
    "stock_option_level": (0, 3),
    "work_life_balance": (1, 4)
}

In [25]:
ordinal_validation = []

for column, (min_value, max_value) in ordinal_ranges.items():

    actual_min = df_clean[column].min()
    actual_max = df_clean[column].max()

    valid = (
        actual_min >= min_value and
        actual_max <= max_value
    )

    ordinal_validation.append({
        "column": column,
        "actual_min": actual_min,
        "actual_max": actual_max,
        "expected_min": min_value,
        "expected_max": max_value,
        "valid": valid
    })

ordinal_validation = pd.DataFrame(ordinal_validation)

ordinal_validation

,column,actual_min,actual_max,expected_min,expected_max,valid
0,education,1,5,1,5,True
1,environment_satisfaction,1,4,1,4,True
2,job_involvement,1,4,1,4,True
3,job_level,1,5,1,5,True
4,job_satisfaction,1,4,1,4,True
5,performance_rating,3,4,3,4,True
6,relationship_satisfaction,1,4,1,4,True
7,stock_option_level,0,3,0,3,True
8,work_life_balance,1,4,1,4,True


In [26]:
numeric_columns = [
    "age",
    "daily_rate",
    "distance_from_home",
    "hourly_rate",
    "monthly_income",
    "monthly_rate",
    "num_companies_worked",
    "percent_salary_hike",
    "total_working_years",
    "training_times_last_year",
    "years_at_company",
    "years_in_current_role",
    "years_since_last_promotion",
    "years_with_curr_manager"
]

In [27]:
df_clean[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
age,1470.0,36.923810,9.135373,18.0,30.0,36.0,43.00,60.0
daily_rate,1470.0,802.485714,403.509100,102.0,465.0,802.0,1157.00,1499.0
distance_from_home,1470.0,9.192517,8.106864,1.0,2.0,7.0,14.00,29.0
hourly_rate,1470.0,65.891156,20.329428,30.0,48.0,66.0,83.75,100.0
monthly_income,1470.0,6502.931293,4707.956783,1009.0,2911.0,4919.0,8379.00,19999.0
monthly_rate,1470.0,14313.103401,7117.786044,2094.0,8047.0,14235.5,20461.50,26999.0
num_companies_worked,1470.0,2.693197,2.498009,0.0,1.0,2.0,4.00,9.0
percent_salary_hike,1470.0,15.209524,3.659938,11.0,12.0,14.0,18.00,25.0
total_working_years,1470.0,11.279592,7.780782,0.0,6.0,10.0,15.00,40.0
training_times_last_year,1470.0,2.799320,1.289271,0.0,2.0,3.0,3.00,6.0


In [28]:
negative_check = {}

for column in numeric_columns:
    negative_count = (df_clean[column] < 0).sum()
    negative_check[column] = negative_count

negative_check

{'age': np.int64(0),
 'daily_rate': np.int64(0),
 'distance_from_home': np.int64(0),
 'hourly_rate': np.int64(0),
 'monthly_income': np.int64(0),
 'monthly_rate': np.int64(0),
 'num_companies_worked': np.int64(0),
 'percent_salary_hike': np.int64(0),
 'total_working_years': np.int64(0),
 'training_times_last_year': np.int64(0),
 'years_at_company': np.int64(0),
 'years_in_current_role': np.int64(0),
 'years_since_last_promotion': np.int64(0),
 'years_with_curr_manager': np.int64(0)}

In [35]:
invalid_tenure = (
    df_clean["years_at_company"]
    > df_clean["total_working_years"]
).sum()

invalid_role_tenure = (
    df_clean["years_in_current_role"]
    > df_clean["years_at_company"]
).sum()

invalid_manager_tenure = (
    df_clean["years_with_curr_manager"]
    > df_clean["years_at_company"]
).sum()

invalid_promotion = (
    df_clean["years_since_last_promotion"]
    > df_clean["years_at_company"]
).sum()

In [36]:
business_logic_checks = pd.DataFrame({
    "check": [
        "YearsAtCompany <= TotalWorkingYears",
        "YearsInCurrentRole <= YearsAtCompany",
        "YearsWithCurrManager <= YearsAtCompany",
        "YearsSinceLastPromotion <= YearsAtCompany"
    ],
    "invalid_count": [
        invalid_tenure,
        invalid_role_tenure,
        invalid_manager_tenure,
        invalid_promotion
    ]
})

business_logic_checks["valid"] = (
    business_logic_checks["invalid_count"] == 0
)

business_logic_checks

,check,invalid_count,valid
0,YearsAtCompany <= TotalWorkingYears,0,True
1,YearsInCurrentRole <= YearsAtCompany,0,True
2,YearsWithCurrManager <= YearsAtCompany,0,True
3,YearsSinceLastPromotion <= YearsAtCompany,0,True


In [37]:
financial_columns = [
    "daily_rate",
    "hourly_rate",
    "monthly_income",
    "monthly_rate"
]

financial_validation = pd.DataFrame({
    "columns": financial_columns,
    "min_value": [
        df_clean[column].min()
        for column in financial_columns
    ],
    "zero_or_negative_count": [
        (df_clean[column] <= 0).sum()
        for column in financial_columns
    ]
})

financial_validation

,columns,min_value,zero_or_negative_count
0,daily_rate,102,0
1,hourly_rate,30,0
2,monthly_income,1009,0
3,monthly_rate,2094,0


In [38]:
age_validation = {
    "min_age": df_clean["age"].min(),
    "max_age": df_clean["age"].max(),
    "invalid_age_count": (
        (df_clean["age"] < 18) |
        (df_clean["age"] > 60)
    ).sum()
}

age_validation

{'min_age': np.int64(18),
 'max_age': np.int64(60),
 'invalid_age_count': np.int64(0)}

In [39]:
identifier_validation = {
    "row_count": len(df_clean),
    "unique_employee_numbers": df_clean["employee_number"].nunique(),
    "duplicate_employee_numbers": df_clean["employee_number"].duplicated().sum()
}

identifier_validation

{'row_count': 1470,
 'unique_employee_numbers': 1470,
 'duplicate_employee_numbers': np.int64(0)}

In [40]:
processed_path = OUTPUT_PATH

df_clean.to_csv(
    processed_path,
    index=False
)

print(f"Clean dataset exported to: {processed_path}")

Clean dataset exported to: ..\data\processed\hr_employee_attrition_clean.csv


In [41]:
import os

print("File exists:", os.path.exists(processed_path))
print("File size:", os.path.getsize(processed_path), "bytes")

File exists: True
File size: 217691 bytes


In [42]:
df_check = pd.read_csv(processed_path)

print("Rows:", df_check.shape[0])
print("Columns:", df_check.shape[1])

Rows: 1470
Columns: 32


In [43]:
print("Missing values:", df_check.isna().sum().sum())
print("Duplicate rows:", df_check.duplicated().sum())
print(
    "Unique employee numbers:",
    df_check["employee_number"].nunique()
)

Missing values: 0
Duplicate rows: 0
Unique employee numbers: 1470


In [47]:
income_quantiles = df_clean["monthly_income"].quantile(
    [0.25, 0.50, 0.75]
)

print("Q1:", income_quantiles.loc[0.25])
print("Median:", income_quantiles.loc[0.50])
print("Q3:", income_quantiles.loc[0.75])

Q1: 2911.0
Median: 4919.0
Q3: 8379.0


## Analytical Feature Definitions

The following analytical features will be created in PostgreSQL rather than
during the Python cleaning stage.

| Feature | Definition |
|---|---|
| age_group | Employee age segmentation |
| tenure_group | Years at company segmentation |
| income_band | Monthly income quartile segmentation |
| distance_group | Distance from home segmentation |
| attrition_flag | Binary attrition indicator |
| overtime_flag | Binary overtime indicator |

Income band thresholds are based on the dataset's quartiles:
Q1 = 2,911, Median = 4,919, and Q3 = 8,379.